<a href="https://colab.research.google.com/github/Yash-Tiwari-12/Text-Sentiment-Analysis_RNN-DL/blob/main/Text_Sentiment_Analysis_NLP(RNN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#RNN for Sentiment Analysis

In [ ]:
import pandas as pd
import numpy as np
df=pd.read_csv("/content/IMDB Dataset.csv")
df.isnull().sum()
df.drop_duplicates(inplace=True)
df.shape

(49582, 2)

#Pre-Processing

In [ ]:
# 1.converting to Lowercase
df["review"]=df["review"].str.lower()


In [ ]:
# 2.Remove URLs
import re
def remove_urls(text):
  text=re.wsub(r"https\S+","",text)
  return text
  df["review"]=df["review"].apply(remove_urls)

In [ ]:
# 3.Remove Punctuations
def remove_punctuations(text):
  text=re.sub(r"[^A-Za-z0-9\s]","",text)
  return text
  df["review"]=df["review"].apply(remove_punctuations)

In [ ]:
# 4.Remove HTML
def remove_html(text):
  text=re.sub(r"<.*?>","",text)
  return text
  df["review"]=df["review"].apply(remove_html)

In [ ]:
# 5.Remove the Stopwords
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
def remove_stopwords(text):
  tokens=word_tokenize(text)
  stop_words=stopwords.words("english")
  for word in tokens:
    if word in stop_words:
      text=text.replace(word,"")
  return text
df["review"]=df["review"].apply(remove_stopwords)
df.head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,review,sentiment
0,e revewers nted wtchg 1 oz epode 'll h...,positive
1,wderful ltle producti. <br /><br /> filming t...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly 's fmly lttle boy (jke) thks 's zom...,negative
4,"petter mttei's ""love time mey"" vully stun...",positive


In [ ]:
# 6.Stemming
from nltk.stem import PorterStemmer
def stemming(text):
  ps=PorterStemmer()
  stemmed_words=[]
  tokens=word_tokenize(text)
  for token in tokens:
    stemmed_token=ps.stem(token)
    stemmed_words.append(stemmed_token)
  return "".join(stemmed_words)
df["review"]=df["review"].apply(stemming)
df.head()

,review,sentiment
0,oneoftheotherreviewhamentionthatafterwatchjust...,positive
1,awonderlittlproduct.<br/><br/>thefilmtechniqui...,positive
2,ithoughtthiwaawonderwaytospendtimeonatoohotsum...,positive
3,basicthere'safamiliwherealittlboy(jake)thinkth...,negative
4,pettermattei's``loveinthetimeofmoney''isavisua...,positive


In [ ]:
# 7.Encoding
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])
y=df["sentiment"]
y

,sentiment
0,1
1,1
2,1
3,0
4,1
...,...
49995,1
49996,0
49997,0
49998,0


In [ ]:
#8.Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer(max_features=5000)
X=tf.fit_transform(df["review"])
X


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4013541 stored elements and shape (49582, 5000)>

#Dataset and DataLoader

---



In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)
X_train.shape

(39665, 5000)

In [ ]:
import torch
from torch.utils.data import TensorDataset,DataLoader

train_set=TensorDataset(
  torch.from_numpy(X_train).float(),
  torch.from_numpy(y_train.values).float()
)
test_set=TensorDataset(
  torch.from_numpy(X_test).float(),
  torch.from_numpy(y_test.values).float()
)
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)

#Build RNN

In [55]:
import torch.nn as nn
import torch.optim as optim
class RNN(nn.Module):
  def __init__(self,input_size,hidden_size=128,num_layers=1):
    super().__init__()
    self.hidden_size=hidden_size
    self.num_layers=num_layers
    #RNN layers
    self.rnn=nn.RNN(input_size,hidden_size=128,num_layers=1,batch_first=True)
    #Fully connect Layer
    self.fc=nn.Linear(hidden_size,1)
  def forward(self,x):
   h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)
   out,_=self.rnn(x,h0)
   out=self.fc(out[ :,-1,:])
   return out
input_size=X_train.shape[1]
model=RNN(input_size)
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())


#Training the RNN

In [59]:
num_epochs=10
for epoch in range(num_epochs):
  model.train()
  for Xb,yb in train_loader:
    optimizer.zero_grad()
    Xb=Xb.unsqueeze(1)
    outputs=model(Xb)
    outputs = torch.sigmoid(outputs) # Apply sigmoid to outputs
    yb = yb.unsqueeze(1) # Reshape yb to match outputs
    loss=criterion(outputs,yb)
    loss.backward() # Add parentheses
    optimizer.step()
  print(f"epoch={epoch+1}/{num_epochs} and loss={loss.item()}") # Correct variable name

epoch=1/10 and loss=0.2153923213481903
epoch=2/10 and loss=0.16689924895763397
epoch=3/10 and loss=0.23027637600898743
epoch=4/10 and loss=0.3219236731529236
epoch=5/10 and loss=0.1827082484960556
epoch=6/10 and loss=0.1565471887588501
epoch=7/10 and loss=0.28042593598365784
epoch=8/10 and loss=0.17574824392795563
epoch=9/10 and loss=0.3771720826625824
epoch=10/10 and loss=0.26697808504104614


#Evaluate